In [ ]:
from flask import Flask, redirect, url_for, render_template, request
import sqlite3

def insert_task(title, assignedTo, createdBy, details):
    conn = sqlite3.connect("tasks.db")
    conn.execute("CREATE TABLE IF NOT EXISTS tasks (id INTEGER PRIMARY KEY AUTOINCREMENT, title, assignedTo, createdBy, details, priority, status)")
    conn.execute("INSERT INTO tasks(title, assignedTo, createdBy, details, priority, status) VALUES (?,?,?,?,?,?)", (title, assignedTo, createdBy, details, "Medium", "Not started"))
    conn.commit()
    conn.close()
    
def find_tasks_indiv(name):
    conn = sqlite3.connect("tasks.db")
    cursor = conn.execute("SELECT * FROM tasks WHERE assignedTo = ? AND priority = ?", (name,"Low"))
    low = cursor.fetchall()
    cursor = conn.execute("SELECT * FROM tasks WHERE assignedTo = ? AND priority = ?", (name,"Medium"))
    medium = cursor.fetchall()
    cursor = conn.execute("SELECT * FROM tasks WHERE assignedTo = ? AND priority = ?", (name,"High"))
    high = cursor.fetchall()
    conn.close()
    return low, medium, high

def all_tasks():
    conn = sqlite3.connect("tasks.db")
    cursor = conn.execute("SELECT * FROM tasks WHERE priority = ?", ("Low",))
    low = cursor.fetchall()
    cursor = conn.execute("SELECT * FROM tasks WHERE priority = ?", ("Medium",))
    medium = cursor.fetchall()
    cursor = conn.execute("SELECT * FROM tasks WHERE priority = ?", ("High",))
    high = cursor.fetchall()
    conn.close()
    return low, medium, high

def update_status(ID, status):
    conn = sqlite3.connect("tasks.db")
    conn.execute("UPDATE tasks SET status = ? WHERE id = ?", (status, ID))
    conn.commit()
    conn.close()
    
def update_priority(ID, priority):
    conn = sqlite3.connect("tasks.db")
    conn.execute("UPDATE tasks SET priority = ? WHERE id = ?", (priority, ID))
    conn.commit()
    conn.close()


app = Flask(__name__)
@app.route("/", methods = ["GET", "POST"])
def homepage():
    if request.method == "GET":
        return render_template("homepage.html")
    query = request.form["search"]
    query = query.strip()
    if query.lower() == "all":
        low, medium, high = all_tasks()
        return render_template("tasklist.html", low=low, medium=medium, high=high)
    else:
        low, medium, high = find_tasks_indiv(query)
        return render_template("tasklist.html", low=low, medium=medium, high=high)
    

@app.route("/createtask/", methods = ["GET", "POST"])
def createtask():
    if request.method == "GET":
        return render_template("createtask.html")
    title = request.form["title"]
    assignedTo = request.form["assignedTo"]
    createdBy = request.form["createdBy"]
    details = request.form["details"]
    insert_task(title, assignedTo, createdBy, details)
    return redirect(url_for("homepage"))

@app.route("/update_status/", methods=["POST"])
def update_status_route():
    data = request.get_json()
    ID = data["id"]
    status = data["status"]
    update_status(ID, status)
    return "", 204

@app.route("/update_priority/", methods=["POST"])
def update_priority_route():
    data = request.get_json()
    ID = data["id"]
    priority = data["priority"]
    update_priority(ID, priority)
    return "", 204

@app.route("/calendar/")
def display_calendar():
    return render_template("calendar.html")

if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [30/May/2026 17:34:01] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2026 17:34:01] "GET /static/homepage_design.css HTTP/1.1" 304 -
127.0.0.1 - - [30/May/2026 17:34:03] "GET /calendar/ HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2026 17:34:03] "GET /static/calendar_design.css HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2026 17:34:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2026 17:34:08] "GET /static/homepage_design.css HTTP/1.1" 304 -
127.0.0.1 - - [30/May/2026 17:34:12] "GET /createtask/ HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2026 17:34:12] "GET /static/createtask_design.css HTTP/1.1" 304 -
127.0.0.1 - - [30/May/2026 17:34:45] "POST /createtask/ HTTP/1.1" 302 -
127.0.0.1 - - [30/May/2026 17:34:46] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2026 17:34:46] "GET /static/homepage_design.css HTTP/1.1" 304 -
127.0.0.1 - - [30/May/2026 17:34:48] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2026 17:34:48] "GET /static/tasklist